# Train Models
This notebook loads the data, extracts features using `utils.py`, and trains the simple, improved, and individual team member models.

In [1]:
import os
import numpy as np
import pandas as pd
import sklearn.model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from utils import extract_simple_features, extract_improved_features, extract_lluc_features, extract_miki_features, extract_jose_features, extract_final_features

## Data Loading and Splitting

In [2]:
quora_df = pd.read_csv("./quora_data.csv")

A_df, test_df = sklearn.model_selection.train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = sklearn.model_selection.train_test_split(A_df, test_size=0.05, random_state=123)

print('train_df.shape=', train_df.shape)
print('val_df.shape=', val_df.shape)
print('test_df.shape=', test_df.shape)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## Feature Extraction Base

In [3]:
y_train = train_df['is_duplicate'].values
y_val = val_df['is_duplicate'].values

X_train_simp = extract_simple_features(train_df)
X_val_simp = extract_simple_features(val_df)

X_train_lluc = extract_lluc_features(train_df)
X_val_lluc = extract_lluc_features(val_df)

X_train_miki = extract_miki_features(train_df)
X_val_miki = extract_miki_features(val_df)

X_train_jose = extract_jose_features(train_df)
X_val_jose = extract_jose_features(val_df)

## Simple Model

In [4]:
model_simple = LogisticRegression()
model_simple.fit(X_train_simp, y_train)
print(f"Simple Model Accuracy on Val: {model_simple.score(X_val_simp, y_val):.4f}")

Simple Model Accuracy on Val: 0.6458


## Improved Model

In [5]:
X_train_improved = extract_improved_features(train_df)
X_val_improved = extract_improved_features(val_df)

model_improved = LogisticRegression()
model_improved.fit(X_train_improved, y_train)
print(f"Improved Model Accuracy on Val: {model_improved.score(X_val_improved, y_val):.4f}")

Improved Model Accuracy on Val: 0.6622


## Miki's Model
Combines the baseline overlap with Miki's TF-IDF Cosine Similarity.

In [6]:
X_train_miki_full = np.hstack((X_train_simp, X_train_miki))
X_val_miki_full = np.hstack((X_val_simp, X_val_miki))

model_miki = LogisticRegression()
model_miki.fit(X_train_miki_full, y_train)
print(f"Miki's Model Accuracy on Val: {model_miki.score(X_val_miki_full, y_val):.4f}")

Miki's Model Accuracy on Val: 0.6788


## Jose's Model
Combines the baseline overlap with Jose's Levenshtein Distance.

In [7]:
X_train_jose_full = np.hstack((X_train_simp, X_train_jose))
X_val_jose_full = np.hstack((X_val_simp, X_val_jose))

model_jose = LogisticRegression()
model_jose.fit(X_train_jose_full, y_train)
print(f"Jose's Model Accuracy on Val: {model_jose.score(X_val_jose_full, y_val):.4f}")

Jose's Model Accuracy on Val: 0.6495


## Lluc's Model
Combines the baseline overlap with Lluc's WH-word and first-word matching.

In [8]:
X_train_lluc_full = np.hstack((X_train_simp, X_train_lluc))
X_val_lluc_full = np.hstack((X_val_simp, X_val_lluc))

model_lluc = DecisionTreeClassifier(max_depth=5, random_state=123)
model_lluc.fit(X_train_lluc_full, y_train)
print(f"Lluc's Model Accuracy on Val: {model_lluc.score(X_val_lluc_full, y_val):.4f}")

Lluc's Model Accuracy on Val: 0.6759


## Final Combined Model
Combines all features: Base, Miki's, Jose's, and Lluc's into a Random Forest Classifier.

In [9]:
X_train_final = extract_final_features(train_df)
X_val_final = extract_final_features(val_df)

model_final = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=123, n_jobs=-1)
model_final.fit(X_train_final, y_train)
print(f"Final Model Accuracy on Val: {model_final.score(X_val_final, y_val):.4f}")

Final Model Accuracy on Val: 0.7202


## Save Models
Save the trained models to the `models` folder.

In [10]:
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/model_simple.pkl'):
    joblib.dump(model_simple, 'models/model_simple.pkl')
if not os.path.exists('models/model_improved.pkl'):
    joblib.dump(model_improved, 'models/model_improved.pkl')
if not os.path.exists('models/model_miki.pkl'):
    joblib.dump(model_miki, 'models/model_miki.pkl')
if not os.path.exists('models/model_jose.pkl'):
    joblib.dump(model_jose, 'models/model_jose.pkl')
if not os.path.exists('models/model_lluc.pkl'):
    joblib.dump(model_lluc, 'models/model_lluc.pkl')
if not os.path.exists('models/model_final.pkl'):
    joblib.dump(model_final, 'models/model_final.pkl')
print('Models saved successfully.')

Models saved successfully.
